# E0: reproduce FLOCK zero-shot results

- **Kaggle:** select the **GPU T4** accelerator, turn **Internet on**, choose **Save Version**, then
  **Save & Run All**. Download the `e0_*` files from the Output tab.
- **Colab:** choose **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. Keep the tab
  open. The last cell downloads `e0_results.zip`.

The notebook checks the walker, runs three small inductive graphs for comparison with Table 9, and
times 20 Hetionet triples at the paper's zero-shot setting (n = 512, P = 2).

In [ ]:
import base64, io, json, os, subprocess, tarfile
from pathlib import Path

EXPERIMENT_COMMIT = "239b4da96a63a378d0a10196d7f9bbddbbae8230"
UPSTREAM_COMMIT = "f35103d25a78bdf4075de5c673a51de4979aa4d7"
WORK = Path(os.environ.get("E0_WORK", "/tmp/e0"))


def running_on_colab():
    try:
        import google.colab
    except ImportError:
        return False
    return "KAGGLE_KERNEL_RUN_TYPE" not in os.environ


ON_COLAB = running_on_colab()
OUTPUT = Path(os.environ.get("E0_OUTPUT", "/content/e0_output" if ON_COLAB else "/kaggle/working"))
REPO = WORK / "flock"
PYTHON = Path(os.environ.get("E0_PYTHON", WORK / "env/bin/python"))
SCRIPTS = REPO / "experiments/e0_reproduction"
RUN_ENV = {
    **os.environ,
    "EXPERIMENT_COMMIT": EXPERIMENT_COMMIT,
    "CUDA_VISIBLE_DEVICES": "0",
    "TQDM_DISABLE": "1",
}
WORK.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)


def sh(command, check=True, cwd=None):
    print("$", command, flush=True)
    bash_command = f"set -o pipefail; {command}"
    code = subprocess.run(bash_command, shell=True, executable="/bin/bash", cwd=cwd, env=RUN_ENV).returncode
    if code and check:
        raise RuntimeError(f"command failed with exit code {code}")
    if code:
        print(f"FAILED with exit code {code}; continuing", flush=True)
    return code == 0

In [ ]:
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True
).stdout
print(gpu)
assert "T4" in gpu, "Select the GPU T4 accelerator in the notebook settings."
sh("nproc")

In [ ]:
if not REPO.exists():
    sh(f"git clone -q https://github.com/jw9730/flock {REPO}")
sh(f"git -C {REPO} checkout -q {UPSTREAM_COMMIT}")
BUNDLE = "H4sIAAAAAAAAA+19bXPbyLVmPvNXoHhrd6mE4uD9xRVurWPLM97YHpfHk9xbuloMREIiRiTBEKBtJXfmt99zuhvA0wApgbKdZCtiuSwCaPTB6T593p7T4Cb5FF8v88tkGS/SZJ5uf/PlPyZ9fN8Xf+nT/ksX3d9YnmNaluU6gUvnLdezrN9cf4Vn6Xx2RZlsieQ2z8u72t13vc3c/ycfzzZm+WqVrsup7USX7jyJ/MR3EicI52ZimVbkz4Or6PJyPr+8TNLQdszBP/qZHz9f7pN+2qTbjOe/+OZr0eD1EATe4fVP31vr33Ft8zfe13og/PyLr3+c/9SMt+lmm893szLL119MHh4w/75nW4/z/3f43DX/s0U6u4k/JsubdDvZ3D6YhrT/7oH5t7zAdlrzb1ueTfLyBfk8+PkXn//hcPgsX22SbWqUi9TYpss0KdK5wbNOR7N8O8/W18bHrFwYTzebdD3PPhlPJ/bYSNZzQ0gI3ZiURpGmc3VfQTdu0qScDAbvdutCdZwsjZ9eLPPZzUS0+cnI1+LK9TbZLIz8ShxsEhLH/1UYL7LrHT2SRWRKceGv6TY/LRZ5aXyXsnCmTLEs6dkmg/d0fZ4UCyK/zNZEPlvLZ7qSvTBzf9mRmKfF2Chy7u9WnFxTd8R6WVEXjzIxnhrP3v5oXO6y5VxdGaTrD9k2X/M6kb0TRyQQxruzp89fnxlZYaTrfHe9IJZ/LJLr9ImxuS0XxGFrDRnnp6c8UoZl2u4FH+W7crMrjcnkm21a7JZyGcr2sbh58nORry8GNFGDQbba5PS4yfaaHrtIq+MFMb/MLqtDvqH6nhfVt80yKa/y7ao6LnaXtNZnaVG3KG6LwdU2X9GYlNyfoc6/pcOa9s/Z+ufEro7WuxUxlRTGelOdKvPtbKEdxNdpvkrLbTarTt8mq6Uk1WowmSdlUtF9Tt9JhM7efm9MxUOM4vgqW6ZxfDKhwcqXH9LRyYRll7TXuX0xePb9mxcvv6XGw2I7i+lsVt5+M8vXJAffsACx/MTlNlkXQsV9SL9Ze5ZNDYt0dblM7Qk/15Aovnr6/uX3b36gns6Hy+wmHY6N4Twrqq9XJErr+Z9pTQwvBm++f34mW77aycvfJesf8mXOX89WJM/5lr/+Rz5P+O9zErjFnzjU4qNni/TjZTKbiUv/NyEfm7+8SrOEuj57/q3semDQZ1QTqJ6jInQybjWAZ62eoNMGmKie7q5+4LE7zao2DTN1ExgL6EwyWjeCUTrcSB+3qpEYqbqNNprtJheDwWCeXimtEFuxWO6jkyfiZlpffxArvtFJiVRdLJKka4zZMimKtDDm+RMjTWYL4yqZ8bLaFUZiXC0z0o6kEXNaD0JblrcbUhLG74z/evdfE169TIVj7IJnVEjNJFvP008jPnli0NoUl8dGTP9Yy4jpvxD3lUm27NzHJ+V9sbiJj9v30UPwfWKZTUqS9Hw7Oq8FXHVEWj9hpVp3Vp3QHuRE9JjOr9NY3NbpVjAnH6NQrbdpuduuxUoeiRN6F1PZwSwpR+fN2TG0mPDAjsyTi7Exz1ZTS0113Q0ziL0IhseK798Zy3Q9qrk9uYC7SXPF63yeFlNuI4a1dbUahGKqsWkbv211q+47UfK1zJN5vKKul5VskWKdsE6l4S7SbTkyyQyV25HQbN+gshqeyGETqvFKGEvRUVGpRGFAB3Jq0xWr9JSmQerkyXt1ZlT1LPUhK0t6ojL9VI5U/1In0p2s8CZFcpXG/NSjqk+6gwZ/O1KiPx1Ke2zRcrre7IrpuUnzkW7y2aKYEjOXNAf0Z3azoaZrMs4VH2r6xVOPJNHzoeBoeHEyST8kNERq1IqESKcjcXEs1txYLQe2l9M31Kv8GgubM32RLItUjW92Ja6wGWaTzm2f1FMpJ2+VrHfJMuZmI/5PPt96MyFrMM9Xk/pC1ZkkY9DYp6JDeYNc2aLHdc4KZF7NMbAreJBezkgy0l0nFycV4/PsOi3KkfS1VGds0NMtzY+y7JNikdieP1LSQYtU9sWLU97YPIS8d7LbzFkUZLuJYIaMZZlf3pZpMdLnR92ySD+ph6mebZFvYvKPy6wos1mhnnEsnKnben1MpaZt9OhrciHZ71rkH7mHQvha8t50PjHe5kXG9xkWT1hxIxXnZTpLdjTWvKpP2Scva2/yilzOrKx1qDg5VjovVmt9bKyTFc2aOphnRE8sXlaaahCu6HnpKy8/+6Qakr2jKZXnulEB1Iu+5tXwyct0lSTpardcjkagTYzmOx2cWvKm5DohNVDKe4RXcsdNc6HfLvN82Uy9XBXMbkvlgwzwgs/m8jp9IWIdgzM2OrYEBFkydt7u6IJ6qi+qc+OKHF9smxZ+zKZfxfveO99vd6nUbJtt+iHLdzSNs92WnTu6Kib9/MnYsJ6cWqR86mP7iZwsFlWYj/ohO501zZuZqJ7rQONKdtnisXwh3awgQ/UhJW9cUFUN//cUpadqSK7Jzcdky/00Aqo6M6ZTw6q02Ujj5veGeTJJ1iysNP2s4Ubd23/PMp0sl5o+SjJ6LIrBSorwz7bkYo2GV/mOAreEB0AGSUppJkZBwdRSLsAqIKrCMOkQDU/k/CTLfM3G41cYRGnnaLJxEpALfk6eOuazfd5SYynuF9IKi1kyqO5snXfUjVIhCV0zbUlCVyR17aUpwr/VQzfkzoZPaG2V2mRMiuyvKTgKw2rK6xYxTTSNHccZQizmMRvEJwYv4tGolpD/oasX/bHVTCIdsVLKPBZLRz1c87itR4lJT7CgURvyIsgvGv3ayOm5mL8L438av4JMqrMnk1WarDXS7b5VN/WdDZF9NHqR+AUYFQwSo4LhexmtHqbDMDzLr/sZ/rU3x11Of93bfb/ekVmy6xSKrJJytqDVFycqwxIndkNrhCpmiprrhGiONL2iX95DfFiSmo1r9REzl9R1QuFvzF6nWFvxp3zbtKllFx/k/yGzJ23CXfEl569IZzsOumNNTcSw0LQLk2K36tuHHEZ2zuSFhJzQ2xUpctIm9UAq7XKu3XqxZ4h0BSEeMCbf4fo6BQmoxOO80T0XPbRNa0p+qbxf9jhFuqc45AIrvV5n6MivUd7aPa6zuFmFdsLFSdbX6cg+UTpX5u2O6k1zxNluH+6+q1hlxkd0kK+XPDScK2QRqFij6IJHsj4ke9++nZx2SV521PQh2VE9qAPtfnlOZNiKWLrVeJs+Lau0THgERjABfP46K0e/TbbXO5E470YATW5tst2tR+dDuoFip+YW8mBmH+dTjtPoW7IpOSMhU4FiQMnhonBNfRVyIcd5UpRzakZ/thQWK3v8bwZnQP9IAroUOc2UluuNsVtvaHlyCoMs/Ixm0+CFTnZ+RsO6NnYb6iNNVnRI9klkczcyv5GV5JXnu+0sFdAwed4yZuSnoJHjQFZ41Ri3jo2hcBFOZeZyeKHCTL6d2ubFRGVQJ9dpORqe/fvbs3cvX5+9eR8/+/7165fvhyeV66PuoWfWw7i6Lx57Us4fTkUKVKTBzp4+HzZOpvagE6lSR8M7gI7hIR9A0iTxkF80VTRPOcbPrjKiRJ7TLI3r1kJbisfk2GlX8DOenlL8PiMtkK3lIYuD9qSaCmJ5U2KpK0CKoOm8DOzxymZHTgfH+UWcfCALygqdGnIwQYNfEKV5TEOfXF1la5qwkamRk8lqal8liSfyTMzKnkZodNJegdRWBrVx1SaOO6u0TutWrSHRu/8+saapNYVGnQb1oqQxrLxcIQMcKFcp8clTtcTeiiujeVrMaKmIODWO5/mMemNdteJQcBuLnN60vvld8vF5c8N36XLzomp6AvQmyXweV2t5NJQp/aGMRKdkyMasJJLdspxymv/OO+WaVxJIF3hlqabiDzcuqoUudDKHopBhkjqJs+bTTmazTjm24j+ZuK2knu0YNbhH7/NzTPYrf9WPQC+oI1hAlf6kGa1Vad0RyhSrjVhG4ayUN0J2szotQop5Ik53PW9ym6jt3kRFe0HJpveY2vbj/aJCUvZPGFiZzIlxpiDYHRs8olw8c1JrMNGDnNdGfQn4Aq6cTD5uszKVWbk7+zV+Zwz/k3XUYECdxzGHQHHM5m0Yx7wW4ngo6ciFcVx5zl34b3WQxpymiBk9eRAMfA/+6/qu38Z/HSuwH/Hfv8NnOBy+26117FfadGlHNzkJPqCvAvVVgDGZY5EPfc/Gxog0DFcAoakEUGpXQ6WTx0qRcV8luZzSO9mtZwt2HWkNyiwrPpJKWLOTKaAXidQOfjo9XSWfTtkZWqbFT0ZByz5lOEZmddkPE/Q+LjK6iXyKXZFe7ZYy55etGNm+3JUi/yGZEpwgUzWgK9WABHX3Lwzj9FQxWBh/fmOF7969XCuc8ckHi6+mHzJ2qjY7OrjkmO+UswmGZfxnrScOAMINHQHuPhAMXubX18RzdSgHqT5KpdOHaHCF4pKCuB8ZPogFizsRCK9ur2zC14CCvzt79se33798814kJSXQ0kh08Y1AV5T7OtmUi+Hg5ZvnPz57//JPZ3EvHDlb1yCy5dcYsuUrEPn9u6dvfjiqw/uB6fdP//DqLI7iN09fC1xYWtrhiz9Y3o3tBChuZOjo/Cl9kWZs2JVIbvLnN9DkzdmrV+0Gb15Bg6r0gi/U38eDXwbvv391Rvw+O6OHMiempTy1auF/ARysEgSK/pd9ILK6CcVf1RXWNQN09bmzsbxhrK420XhK7ictd1ovGP214aKDOJK8cBf4pFDL3TzBBjGnTmQjgBOlrIzkA8vUPy+PmK0Se5usWMaG0CmxyFA+qf1CoQDJ6VXetEqu8lopBQwzGj4ZVi4q6b2psU9yyfOoYHB2POrJl/hYe+VUGCM7N7pDqDohAaoerbnGyCJLFjl7Q0FQqkumx4OkaFGIsFwO4TYJQ9KNJpy8FB5kpy0jlOwCktA1CkJz9IiuGigMPQUn50N1ZchwhfouWb3idDhPzgRnS8k1D+u46mSq/p5UN07UQExEPdFUPFs9tU0rUg7ZesIyGjezTM2bAxRsumMs6NYlD0sKYTjDKCMDcb0WDtErHx0uhSDTfrPOP65FzUNhpJ/oz1Io+59ogZEbuiYlevuTMc/J/ArrukiLunSiqLE7DvCZm6S4OR+yHA5FomZYq52X66uUlPgsHXaSKnoRAX/2FxLU7EywpKA5S/4eRcJxc3FPbUHde6e+QO+cLx/qm69dtPpsqg6ae+pzTdN+xROV5Bxga1+lxOFbBCeD7nOqOzgCaz1ojdyTkyVjx5on8vCST7Fyy5RUratjRtEOT4aM886ti0peoKcqK8QwGJ4WaJs66IhNTUt2qFWusNImZ3U1qm8/OX8CPasUqfAitccWSTMVYMvLXU7u5pPhKyl9hzoRBUSH+uCL53B/k3ikbtTECEc8/pAsd2khgVg5OCW7tjwNIu5kjVXUJSNankx2kKyT5W2RVYfybuGHDveUlmzzj2xi+Ax/ZV3Af7loUtx4PoxozdO80tnz4byQCkDzbM75WfUCor8NV9stp4j5phdDWrHDRVYWllmfWwwvqgQR15XsuPhBGkqOukXIsWuEURpS6QOqJICIRKZVO9GK/WWRXVKOMycvX4lzo2FqKqvZdhaaBEKlvMcqfImVgT1k0cWdYNblsbLt4gAMvMpFSxMpbQ8LhzzDCr7JDEnhFU1EUWusTiozIO6Yyj/KM6lEjoFqtf7ti0mZj7CVMAixNAjT3iam3Us1WSIFtU+TCMZRnei5MJhDmc6JifKUjYw4OmkaS9vMWSIGwGblSOoAUfjUeAOsujYxdapqWyhaG1IAmWbXi7IQiIUqPWoKmbTnkaRaTFZZobZL065SEs4gxTM0kZs0IbbSVb69Fc9cJwFFbQzrhYy9uHR7RaK0I59iO+pToiRjq0oHy1lSC6CeicahlAtgrCY3lZIzhYkfqwUaq34hIVikJOKi2HHPkxqnko9BxzTUT3GfaeBym5i9+ynef4RtWckyJRa8Sr1IbE/xci5OkgUHXdNqwOf/D124OFFpQtaQIoPb1rwHEIbGHxYqYE+qH1RHK19axJyxryVdFJxxWhZ9YxVAxhutpRgiudIKdJpr7UKtW/oGc/t8txpwGte44aHP0B/qqJ53HovqLDSuppsnQX1FAENaKj5f1VQwmsFnoVU14yIdLb/i1S2D3le1/1m3UYJAMis6VIc4Gxmr/rjMlyl5FOLe5LIY3X3/ifF7snt14EwqopbouuhQS2CL5SQS2OJb91oszLMYuaYZ2fR9wynUi8IoKzVzfSlAWJkGYmuy3XHqa1SfevfjD0+/PYt/OHv1giz/jtz+T9uCKTDM8dvf2u3+KaLT+tYi3k/VJZo11rYUG5/UXTkHVWZ7cP6JUaEq+hk2eNDeXM24m5sZd3Ixd1MSgzRkqDinL8X0XNktOWYX8ADixJ2dNZnJ/ZDWnTdDOhbu/krgGQ/wKftK1e1vRahd3b/PpeVbvrm5vnsEFCgHnfbG53bSijWZLxV1XOXsG3QxsPryZJNvRpXab06fQ9KyKle46BYGN/6L9MirKl+tprfpU8nLRR0IiTXZ+I8CZBqZJ/esQiFkbfQPQT+mRkJIOo0V1/mFtJIcEzABDgpqn1ekCYrNkmHz8VArqRBdn8teLipA/04nvykGOATcVd2di2rSGmzjmGBXLMCNEYN2ANSrZEJd+CeB9I763BHufTEaD9n/6zr24/7fv8PnmHD/oTTuxn9pqt3O/l9bXP+SjB76/IvP/98Gw4hV84C+CO+SUSPbG44HQ6GM+UzthxikaLd86UdOt0+cMOTvC3Hgu/T9vTwfOfxdnffo+wvx1TW5zQt13ncHv4w1sp7Zj6wDZD3Xaeg6bkPXc4OasOPZDWHP99uEg378uqYD/JoNw65pAcOW1XBshcCxbbcJW2Y/ll03wpG2G8qOD5QdGGtk2fcdjfKf/9h7ii0fxtppCDum1xB2I78mbAPHbmS16facY8sHuo7dDLW8oOg6llPTtcRIKMJ2ELYJ95xjR8hrzXDQMOyHIFxA2AlRuJwO4b5zbIkVpCjbod+wHAJlO4oalsOgoeyY+np68+q053IS4lRxbMNyCmGKPTcCji2QLdNv0+0rWxEOtR81hIMACIdA2PVgqKM23Z6y5ZoB0G1mWKqnmt+wIeuDaHm+16bbW7Q0vQWi5YH68OxmETsWDLRndya4t/oILFAfkpq8EKL6CJpVrKuPSFdcr99bRpl86kXatkGqldJQTINUK3JK9IG0Ezod0os0WfKm9T7UIxhyci1AxHykDqYiBBFzQ6tF3Tby7XUv0iZKt+U10m1GMNuW14w5zLXlt9m2jWKW9ePZQ5494NkGs+z6KGaos72wRdqhGKfsSRrMlGs21sIOYWm5bmMgHRO4dluWmUlTBLlNehH3LSAeeI2k+R7osSBq5toPwR8J7fZcu/2HPAAhl7Za8R3hiB/iuj3X7jEi7uMCC8Bg+ahFKfwCthviLTv5mivEk1U/ph3N+wM5c0HAQwt8kgD0aOR4LW+IU1791FkIS8v3wBsK0PFsxtszQaUEbSdM1DftpQokUZH4XmOcPQuEyw8bNQZqxI+6NO37aXoWGOZmXqWBqOa18UM8B0iSIHRIOn3Y1AxFM6ee5uH6jUn20ET5YXtOTz+4PaiG4OwpsVWPg/MZBjChyKupm6eqpu0eqr4LUxpI705NF6zaIGqUlR/ByglNp0O1x6SSK45UQUsFYA+DEKnCqgmiDtEe0yqtXaWfgkZ6ZXxSuzwQuPjgWvpm2KHaY1plIFSxCqZfjnzFqt2IsO85wKrdcfD6TGuAyiGyGnc2NMHkh1EIVHFaW9qYqfZZq7aPvMJq9VGY/IaqF6IabmkIptpjXqXyqak2itBD5ypwQUlEqPu7rPZZrQHaWQuIOkgU/AvPw9Ua6kb+u9eGdXM/VRNdCzNqqJqB3VC17GaATR94tayoTdXpQ9WBUMEEzWTitFpWBFQDpBq2qXq9qLpI1QKqyKsFRF0k2hlgopJd5z2mFuiqRSrPa1kGFyy6DQvWb8W+L1+9fWb8sEpECeZ9GsoEW+diTsdEp9UDp9WE6XVbwaAg/Yox6PtJ2+izQqbBMcF/ckGwdHe5xTQ7MVGkB4TvobAbw1BMrQDLLkq0bzey5UaYP/O8DmG7JdP7CduO5qU306x5q+An2w7MshO1jMFzfm9Euu1BWGZKKsJRQ1g6ERXHJnCsxb864Wf5epZuyjdpSWFwH75NJG/5jQqx0M1wQLxtVwuMnBb952ef9sj3gdkOgLjvBwdmuxl0F3kn93oP7a6AHxh39NRdTLQ4mMTzMdECDrPbytT+x9Nvv3cssw/THrqTVuMBuCaImm8D1xaKeKATfrrNijL/k9uDsoVzLTMKVTINRE1m2Sp9gnPt6nP9/A+bdJ4lPQXNwWDBQ3fLRj1qNmYZ81o0HLr+bjZa3LuyPTBXTgBxaIBLG3ItLviWTmC1vHfLuzm1nWBkmSd9qKND7cDyllSq9RU5+8m7Zjs9XpG3+5HHFaYslryAthp8ITtwcMbbcERF3etF3cEwXHqDVdZFS+5B6sMCA+axpzC4oBOizOyBcAzCEw6k2g7iMSh2XVikNx6DrDvNvN+Bx4CT5HUi1b5wDFAFaTsMxtjIrtOm2h+L0bwksNs6FgOr28dh7gAEfafXtnCYYXrRaoN4Y7TqtbL0RyAxGirR6NGDOIwNc2t3qPbNlaMGBW/wEAijibLndcj2B2FCZPd4EKYDhvRcQ+iFepChPwjC6NCi1YmVe0qVDoZEoLcOgTC61rA7DPdFYSyc4gh9A+QYPf4IvUHfbhPuK1qYrPX6oDAu+CSe+3AUBhPzMvdcoTDoAkLCSSbGqsDZ0ikfg8I4YBlcwCKcCCfZbGIc10Lb2MpOHAnC6N4nDPhBDAbjHFJpD8ZgLBOm2oLoPYLFLCMBtZhNjN6D9oD3B2EczH25uJ4dTcrA/9RARttrke4PwjimluxrlKcdoh+EIAzOdQdwOwqDQQc0NBGMABGH5GaAUXzotKGn/hCMg7UZbgBOiFaoAAl66bHV9jHqkD5CxAMUNJmBfK8uAN/ygmIc8+atrO4xKEwU1SNuTcyabtQsbHn6hTpdx3V0+uH4C4Z1PsDJmMP2QXV7DnJrdSp/+iR2PRNrYCDDehCBcTHZabeDm555XZzYhtODIIyWwfa6jPbJ6qIwSbzgPhAGi278qOPI90rrRgjCoHU6CMLgnD4Qg8FkfRA1rAa4agNI1gcWeB+hZXWo9sFgtAw2hMl+qK1W0FKIH0Zeh2gfaA0tgudiUkDz8MCn1dJ9raxAbwwG5bdJg/g4vHazZnwfU/Wu1fZ1eiEwmhEAVRxigU0IPnTgw/hGrTxfXwQmQFECpeQFsGoCmSlXYwNUw67n3mutanAeFB3oVL2GV9/0kerDIBgfBdgBDYE+bBCALgw0XvV57QvBYKEYgBIHAZgAK1pa2dPeAExD0oQamn7wS5dmL/jFBkkyARg2AxepItSEnHaI9oZfHMwROw27h/EXLQh0dMrH4S9ahNALf8FCqZalOwZ/kYnXOm8HrjLIsjxfOW5AuK2iesMvujsBBR2mNtYA2KIP47erAPujL5hhcCGLg3Q950BBbYvsEeCLa2GgbyPDGtzUyBbWk3huG/w4Enxx0PrZEIFiihCUs16y3UZ+jkJecJ4ho+NiTZpW++gjqOnuo90beXEPRSVYAuFCQNRCXnSj3x958dAGo9+KtVnKrVE3HCyROgJ20fnFbJKrlYtjLTEMtmd/DuwitWRdng+JWQ+yHD5YCxeLXFq0j4BdnAiTd5gAj1C6HZBuhLnai/po3MXD6kOrmWzbQ90N2R0bYZfPRV200sdGn9lYi4fZBhurLtvA/fGwC9aQQ37asUGreeDhOhZiD1YbfFDkNdJvt6l4/RJvU60Ja8XcuPkFE+Oeh5gqSrrTjpd4B/C9VBFhk659FQOjLgPMAytv/FZaXKqy17TCdqv7+UV0TUW/il8MXxDfwvjQ5aBU4lv8WrkH4Vsy9qyHvHmEUAsW7WaZhREGi05b1Pqmqn0MjWWcoaJUTHeAUsPSrrCz6adnntrXUGQILUIU7QBgtcgECQs7gGbvRDXiiAGsqQihzLBRJ5EWvnV3wPSc31CzWY3tCBGyD6FcMPK0sDFq0+05vaF/IAkRItoUgr8vB6Ia6NYM9we5IkwJhCBXkYWliqFzYKRbM3wEzBXiUgqhhi/CoqcQUkwROkWR0ylX7LmSMMLxTMxHaMAepAbQUnpBB3zprTs0dd0QDlz0fq1GdwQ2ZA99p4M29dUdGt3G7w1wU4IP7n6IyEvQzYX0lK0At5/4UC8ZhIeUlqOlLjvpkL6yFaAPiJYxRIc7gC2DkYMgQNjOxPeHuSKt9hdUSBTAaEcerGQEyaPPw7kCDaCHDFSErlAEGG6EiGbUQV76A10RemERwE0RbvCKIFHdIv1wpCvC2vIIufZd5NoC0h6OeXtvWX+kK0QcIoASkBBB8xCg3AihXNJ1nwF1SVVc8Q3lGOjzRybQDiMc8QcjXRGCuWGEA45z7TaqTEpHRdlrc30U0hVqhe5Wo9AiBy1HhEvsDhk/AumyEIppkC4bt9OZWO4DdG27Xe7TG+zSfM5GmQV6KV8zzboa3bMNp9fWFHQ47QOwiIfKG81k0OG2Hyyi7cdtALbAcpBoQxPD58Dv4ms98ucaFONB3lG3yyDMWklE2KHZBxPRChLCRm0EuIB8KJQLsVIwiLp7cHrMaYjZCdiJGkSaAwIaGvHp0OtuwekxpaGHrjx4H6ibPcAJdKLt1wb0Q7oCfX81CBImnzzYCyN1R+0APGi3Ee4hCyH502L1EM0u+tNnneKcqg3N8gJmnGQ6vZJeTLM5XXytx6QGiMPg9kc9HAWXMsK4sL1voB/Q5SPk5LmoCNGRtRpRChGp3cdqn6Wq1bKEIMBoZbFkKsTdrWHLwh+DiEjTXQkUlHlGFmBPIa4dTLZErXqDYxCRUIMyo4brELkOYWdIhBsVIrOz66gPvOejZXfh9QSqBEFdgBDJx9JHr02zD7znRVrNNiDxuGpdCL31eNDt7OrqA/D5OLUuLCCtSgm3/ARYwutFD0b4fPRYPSjECz3M6YDpiTzcM20/GO3CnI4Lu8Q9VBpYhKdtjvcevNmopTfA2HpobIFwqNUKtco8jwC8fP11G00wqG+vAtVha57bZ+42snCqzbDRH5gvxT03QaTlavfATn0RLxhx2PmiYjMMWKrABJNKQZduX7QrxNUcwla6CCHG0AEjjI5Vq66mP9ilucsu2v5Q81zB4cDShKiFbR6Bd2nQi21i2TYYCRv3wkagM91WqdZxeJf2piIojggsrXIbDBTuOAlbDvsReJevbfiAeQ5xW5d0Ayv7BIQPIT790C49kwe5+AhBAIgDI6yRCIN2cdxxcJceI4HHFyKQHWCGSU9StyOz4+CuQAPxIfYPsZY6BC8sQn8ocmsUhn9f+IEoDE4+VCceRGEirca3u93n66EwmKV/KAwT4BL3AdzVYBikG2Fpb9itAe2dpMfdkwHkjg/jMOiAOh08pC8OEyE44KAax7Qa5G81GfM6qMRn4zAaAASChRu62iv7S8AwmNKCGEMfZ/fBb3yLMKMVQnjewn/AVAdjfTF/YRAGs0vodGulT2YHC+kpWAG+TcG3IXDVtgObjWQFuFGQzM4DQZgQS8wCeHFjgBV9PgBtrXxaJ3h9EAoD9T+4H9cPwAfFQC7owl29dYf2qsrgQGo8gO2CrXxIFwnpDcKA+xlBHXmEBhrVpZYg7iTGj4JgItxGF8ImgTsgGMyNh23qR0Aw6O9HYBejELVIEPYn3R+CCZA0IAIgZREqzi+FwESY6gsBdwqR6dBGdwgoW5/zvrcIXxEauTjXwDW8WDDCFylFnW1lRyAw6BJEUMwYYWipkcYqt8jvgj/HyDi6BREiMDrWFx2a7tZ77o6AYLSiJwvKBmwMak30/TSLFT34lW9a9REAu3rK3gHnL0Cn0+yCE33Su+jrW31QGO1FR90N9f02pyBVtwcMg1k4cv0fBsNgXAGvtzsMw2jj2x3ePsldzAwdmlJEYbTXNZrd/TB9UBjcvB946Alg2R46e1gsGe55/1ofGAYD5sBHXlGSAnRtcVL9B7317TAO42m8AlVNOXf3VvWCYTRMHov00NXyYICxLjKMupt/+mw5wnd/euDgeWABHahBCFEht1dqTxymlWCEZC6+zggnVQsMH/TGNz3VBd5VoNU4u4g4ae+Z64I/PSYVC/NcUPt6JSKmFLVNTu6el3P1RWFsTCliRc9hFAZNrbXnlWRfD4XRSXcQka+DwgSIEHjtrN5Xw2GwOq5dOv/1YBjEJKIOq31hmAAHGeuoQ3whVgBv7I0QAIrau/Z6wzC4WxA9ioMoTIDVgF6rTOsIFEZLmsJ+EL0MEdKWoa/5Tw/edhRoP5kAPhTmsPANYCG+gDtsLaKjURgJSO+r9bBwf4BnHaK/551vvWEYxOnBTdfr4iACx3xl5H8GENMKyxrSJhY0u5hX+hI4THSgMC3UaqXgvcx6Okv3XL8ECuPiez4cfKcu7k0gJ/troDCYlUZAF1HGtmd3BAijJ9MA3ozQKIewoS/CrWVh553UXxuG0SovD+14+lwUBt80F2CuBd/xGzmHXnTXE4WxtW04sEECN8uGCAJpgt6uWuu750jLJgYmAgLokSC8ilbLa0dMvbYcRZaWpAbTgeCqD5vbdDzZ3wNk99xzFKFXH0KCx9HwD9QnWqGrzWDX4Jd/9E8s/VN/8Pe/tulfdtk2FQen15vdpPx0509e9f3c/ftfpmV5Xvv33wI3CB5//+vv8Pk349u3Pxrp+kO2zdc88+LXGcsF+cuNZIzFj0rnu9J4e/uefy9yLH828rSYiV9BNZL1XNzDP0HMvxf9cZHNFsY6TefGsx+fPz0tNuksu8pmhvj98WIymKXbkk5Mp7Zp+xPLIrM9mC34BzXL0zX/uOoy+2u6nU6diTdxBrNlvptvstnNMuVTFp1Kk+KWf8R7OrUmlju4ypYp/xD4dOpO7AkdF0xSdR9NzEE2Xyd8L9H5OVv/nNiyI3/wc365zC65G5+arZLtzW5TJFeCEDkNg9VmlZQLvk7Hg3Wy/bhIlgX1PLG5XzLWH/PtzSduHgzWu9XmltsSVXewSWY3Cf9uPLX2qatNwb/BPp0G9Ij2YHN7ma3nliUfxKRj/kFR0dohWtT89jZZUXN6LuqMF2dalIKyQ0wOill2k5WnyzTZrplkNLH4nKRvBXyUlrtNmef8uEEoiBS36gEtl47KBf9Y6IZazMolkw35pJjZ6zSXv3LN9ETbv8xXPLqBST2Xtxt60NP0U5muC/6xZb5i8QDutksaTofv4s4+LtKUembfm47+0cL++Hn8PH4eP4+fx8/j5/Hz+Hn8PH4eP4+fx8/j5/Hz+Hn8/It9/hs0DtJZAMgAAA=="
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE)), mode="r:gz") as archive:
    archive.extractall(REPO)
print("code ready at", REPO)

In [ ]:
sh("pip install -q uv")
sh(f"uv venv -q --python 3.12 {WORK / 'env'}")
sh(f"uv pip install -q --python {PYTHON} --index-url https://download.pytorch.org/whl/cu126 torch==2.8.0")
sh(f"uv pip install -q --python {PYTHON} -r {REPO}/experiments/requirements-gpu.txt")
sh(
    f"uv pip install -q --python {PYTHON} --only-binary :all: torch-scatter==2.1.2 "
    "-f https://data.pyg.org/whl/torch-2.8.0+cu126.html"
)
sh(f"uv pip install -q --python {PYTHON} --no-deps .", cwd=REPO / "graph-walker")
sh(f"{PYTHON} -c 'import torch, graph_walker; print(torch.__version__, torch.cuda.get_device_name(0))'", cwd=WORK)
sh(f"uv pip freeze --python {PYTHON} > {OUTPUT}/e0_gpu_freeze.txt")

In [ ]:
sh(
    f"{PYTHON} check_walker.py --output {OUTPUT}/e0_walker_check_kaggle.json "
    f"2>&1 | tee {OUTPUT}/e0_walker_check_kaggle.log",
    check=False,
    cwd=SCRIPTS,
)

In [ ]:
sh(
    f"{PYTHON} reproduce_zero_shot.py --device cuda --batch-size 8 --data-root {WORK}/data "
    "--datasets FB15k237Inductive:v1,WN18RRInductive:v1,NELLInductive:v1 "
    f"--output {OUTPUT}/e0_zero_shot.json 2>&1 | tee {OUTPUT}/e0_zero_shot.log",
    check=False,
    cwd=SCRIPTS,
)

In [ ]:
sh(
    f"{PYTHON} reproduce_zero_shot.py --device cuda --batch-size 1 --max-triples 20 --data-root {WORK}/data "
    f"--datasets Hetionet --output {OUTPUT}/e0_hetionet_timing.json 2>&1 | tee {OUTPUT}/e0_hetionet_timing.log",
    check=False,
    cwd=SCRIPTS,
)

In [ ]:
for name in ("e0_zero_shot.json", "e0_hetionet_timing.json"):
    path = OUTPUT / name
    if not path.exists():
        print(name, "is missing; check its log")
        continue
    for run in json.loads(path.read_text())["runs"]:
        print(
            run["dataset"], "| MRR", round(run["measured"]["mrr"], 3), "| paper", run["paper_all_relations"]["mrr"],
            "| full run", run["full_run"], "| s/triple", round(run["seconds_per_triple"], 2),
            "| peak GPU GB", round(run["peak_gpu_memory_gb"], 2),
        )

In [ ]:
if ON_COLAB:
    import shutil

    from google.colab import files

    files.download(shutil.make_archive("/content/e0_results", "zip", OUTPUT))